In [ ]:
a = 3
b = 5
tot = a + b
print(tot)

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
dataset_candidates = [
    ROOT / '26_29_09_2017_KCL' / '26-29_09_2017_KCL',
    ROOT / '26_29_09_2017_KCL',
    ROOT / '26-29_09_2017_KCL',
]

DATASET_ROOT = next((candidate for candidate in dataset_candidates if candidate.exists()), None)

if DATASET_ROOT is None:
    raise FileNotFoundError('Could not find the dataset. Expected one of: ' + ', '.join(str(c) for c in dataset_candidates))

print('Dataset root:', DATASET_ROOT.resolve())
print('Exists:', DATASET_ROOT.exists())
for child in sorted(DATASET_ROOT.iterdir()):
    print(child.name)


In [ ]:
from pathlib import Path
import re
import pandas as pd

audio_files = sorted((DATASET_ROOT).rglob('*.wav'))
print('Audio files found:', len(audio_files))

records = []
for path in audio_files:
    label = 'HC' if 'HC' in path.parts else 'PD' if 'PD' in path.parts else None
    filename = path.stem
    match = re.match(r'(ID\d+)', filename)
    participant_id = match.group(1) if match else None
    task = 'SpontaneousDialogue' if 'SpontaneousDialogue' in path.parts else 'ReadText' if 'ReadText' in path.parts else 'Unknown'
    records.append({
        'path': str(path),
        'filename': filename,
        'label': label,
        'participant_id': participant_id,
        'task': task,
    })

df = pd.DataFrame(records)
print(df['label'].value_counts().to_dict())
print('Task breakdown:', df['task'].value_counts().to_dict())
print(df.head(3).to_string(index=False))


In [ ]:
%pip install librosa

In [5]:
import numpy as np
import librosa

FEATURE_SAMPLE_RATE = 16000
N_MFCC = 13

def summarize_feature_matrix(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim == 1:
        values = values.reshape(1, -1)
    return np.concatenate([np.mean(values, axis=1), np.std(values, axis=1)])

def extract_voice_features(path):
    audio, sample_rate = librosa.load(path, sr=FEATURE_SAMPLE_RATE, mono=True)
    if audio.size == 0:
        raise ValueError(f'Audio file is empty: {path}')

    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    delta_delta = librosa.feature.delta(mfcc, order=2)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sample_rate)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sample_rate)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sample_rate)
    zero_crossing_rate = librosa.feature.zero_crossing_rate(audio)
    rms = librosa.feature.rms(y=audio)

    feature_values = np.concatenate([
        summarize_feature_matrix(mfcc),
        summarize_feature_matrix(delta),
        summarize_feature_matrix(delta_delta),
        summarize_feature_matrix(spectral_centroid),
        summarize_feature_matrix(spectral_bandwidth),
        summarize_feature_matrix(spectral_rolloff),
        summarize_feature_matrix(zero_crossing_rate),
        summarize_feature_matrix(rms),
    ])

    f0, voiced_flag, voiced_probability = librosa.pyin(
        audio,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C6'),
        sr=sample_rate,
        frame_length=2048,
        hop_length=256,
    )

    valid_f0 = f0[np.isfinite(f0)]
    voiced_fraction = float(np.isfinite(f0).mean()) if len(f0) else 0.0
    if len(valid_f0) >= 2:
        periods = 1.0 / valid_f0
        period_variation = float(np.mean(np.abs(np.diff(periods))) / np.mean(periods)) if np.mean(periods) > 0 else 0.0
        pitch_features = np.array([
            float(np.mean(valid_f0)),
            float(np.std(valid_f0)),
            float(np.median(valid_f0)),
            float(np.ptp(valid_f0)),
            voiced_fraction,
            float(np.mean(voiced_probability[np.isfinite(voiced_probability)])) if np.any(np.isfinite(voiced_probability)) else 0.0,
            period_variation,
            0.0,
        ], dtype=np.float64)
    else:
        pitch_features = np.zeros(8, dtype=np.float64)

    return np.concatenate([feature_values, pitch_features]).astype(np.float64)

sample_path = audio_files[0]
sample_features = extract_voice_features(sample_path)
print('Sample path:', sample_path)
print('Feature vector length:', len(sample_features))
print('First 10 values:', np.round(sample_features[:10], 4))


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

readtext_df = df[df['task'] == 'ReadText'].copy()
readtext_df = readtext_df.dropna(subset=['label', 'participant_id']).copy()

feature_rows = []
failed = []
for record in readtext_df.to_dict('records'):
    try:
        features = extract_voice_features(record['path'])
        row = {
            'path': record['path'],
            'participant_id': record['participant_id'],
            'label': record['label'],
            'task': record['task'],
        }
        row.update({f'feature_{index:03d}': value for index, value in enumerate(features)})
        feature_rows.append(row)
    except Exception as exc:
        failed.append((record['path'], str(exc)))

features_df = pd.DataFrame(feature_rows)
feature_columns = [col for col in features_df.columns if col.startswith('feature_')]

print('Feature rows:', len(features_df))
print('Feature columns:', len(feature_columns))
print('Failed files:', len(failed))
print('Missing values:', int(features_df[feature_columns].isna().any(axis=1).sum()))
print('Label counts:', features_df['label'].value_counts().to_dict())


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

participant_df = (
    features_df[['participant_id', 'label']]
    .drop_duplicates()
    .sort_values('participant_id')
    .reset_index(drop=True)
)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(participant_df, participant_df['label'], groups=participant_df['participant_id']))
train_participants = set(participant_df.iloc[train_idx]['participant_id'])
test_participants = set(participant_df.iloc[test_idx]['participant_id'])

train_df = features_df[features_df['participant_id'].isin(train_participants)].copy()
test_df = features_df[features_df['participant_id'].isin(test_participants)].copy()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = (train_df['label'] == 'PD').astype(int)
y_test = (test_df['label'] == 'PD').astype(int)

print('Train participants:', len(train_participants))
print('Test participants:', len(test_participants))
print('Train recordings:', len(train_df))
print('Test recordings:', len(test_df))
print('Train labels:', y_train.value_counts().to_dict())
print('Test labels:', y_test.value_counts().to_dict())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    max_features='sqrt',
)

model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

metrics = {
    'accuracy': accuracy_score(y_test, predictions),
    'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
    'precision': precision_score(y_test, predictions, zero_division=0),
    'recall': recall_score(y_test, predictions, zero_division=0),
    'f1': f1_score(y_test, predictions, zero_division=0),
    'roc_auc': roc_auc_score(y_test, probabilities),
}

print('Model metrics:')
for key, value in metrics.items():
    print(f'{key}: {value:.4f}')

from sklearn.metrics import confusion_matrix
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))


In [ ]:
from pathlib import Path
import joblib

output_dir = Path.cwd() / 'data'
output_dir.mkdir(exist_ok=True)
model_path = output_dir / 'pd_voice_model.joblib'
joblib.dump(model, model_path)
print('Saved model to:', model_path.resolve())

features_df.to_csv(output_dir / 'readtext_features.csv', index=False)
print('Saved feature table to:', (output_dir / 'readtext_features.csv').resolve())


In [ ]:
# Quick inference example on a single file
sample_audio = next((DATASET_ROOT / 'ReadText' / 'HC').glob('*.wav'))
sample_features = pd.DataFrame([extract_voice_features(str(sample_audio))], columns=feature_columns)
sample_probability = model.predict_proba(sample_features)[0, 1]
sample_label = 'PD' if sample_probability >= 0.5 else 'HC'
print('Sample audio:', sample_audio.name)
print('PD probability:', round(sample_probability, 4))
print('Predicted label:', sample_label)


In [ ]:
# Compare one healthy and one Parkinson's recording from the actual dataset
hc_candidates = list(DATASET_ROOT.rglob('ReadText/HC/*.wav')) + list(DATASET_ROOT.rglob('SpontaneousDialogue/HC/*.wav'))
pd_candidates = list(DATASET_ROOT.rglob('ReadText/PD/*.wav')) + list(DATASET_ROOT.rglob('SpontaneousDialogue/PD/*.wav'))

if not hc_candidates or not pd_candidates:
    raise FileNotFoundError(f'Could not find sample HC/PD audio files under {DATASET_ROOT}')

comparison_files = [hc_candidates[0], pd_candidates[0]]

for audio_path in comparison_files:
    features = pd.DataFrame([extract_voice_features(str(audio_path))], columns=feature_columns)
    prob = float(model.predict_proba(features)[0, 1])
    label = 'PD' if prob >= 0.5 else 'HC'
    print(f'Audio: {audio_path.name}')
    print(f'PD probability: {prob:.4f}')
    print(f'Predicted label: {label}')
    print('-' * 40)
